# E03 · The Kalman filter - gain, noise tuning and innovation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/06_estimation_perception/E03_the_kalman_filter_gain_noise_tuning_and_innovati.ipynb)

**Track:** Estimation and perception · **Skill axis (sheet):** Estimation and perception · **Time:** about 60 min

**Skill Hub concepts:** Kalman filter; Kalman gain; Process noise (Q) and measurement noise (R); Innovation

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("E03")

## Map: predict with the model, correct with the sensor

```
  predict:  x = F x            P = F P F^T + Q          (uncertainty grows)
  update:   y = z - H x        S = H P H^T + R           (innovation and its covariance)
            K = P H^T S^-1     x = x + K y      P = (I - K H) P   (uncertainty shrinks)
```

| Symbol | Meaning | Tuning intuition |
|---|---|---|
| $Q$ | process noise: how wrong the motion model can be per step | larger Q -> trust measurements more, respond faster, noisier |
| $R$ | measurement noise covariance (from the datasheet or a still recording) | larger R -> smoother, slower |
| $K$ | Kalman gain, the weight on the innovation | scalar case: $K = P/(P + R)$ |
| $y$ (innovation) | measurement minus prediction | should be zero-mean and white if the filter is consistent |
| NIS $= y^\top S^{-1} y$ | normalized innovation squared | averages about the measurement dimension (1 here) when Q and R are right |

Constant-velocity model, state $[p, v]$, white acceleration noise of spectral density $q$:
$$F = \begin{bmatrix}1 & \Delta t\\ 0 & 1\end{bmatrix}, \quad
Q = q\begin{bmatrix}\Delta t^3/3 & \Delta t^2/2\\ \Delta t^2/2 & \Delta t\end{bmatrix}, \quad H = \begin{bmatrix}1 & 0\end{bmatrix}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Task 1: a constant-velocity Kalman filter

In [ ]:
class KalmanCV:
    def __init__(self, dt, q, r, x0=(0.0, 0.0), p0=10.0):
        self.dt = dt
        self.F = np.array([[1.0, dt], [0.0, 1.0]])
        self.Q = q * np.array([[dt ** 3 / 3, dt ** 2 / 2], [dt ** 2 / 2, dt]])
        self.H = np.array([[1.0, 0.0]])
        self.R = np.array([[r]])
        self.x = np.array(x0, float)
        self.P = np.eye(2) * p0
        self.y = None   # last innovation
        self.S = None   # last innovation covariance

    def predict(self):
        raise NotImplementedError  # TODO

    def update(self, z):
        raise NotImplementedError  # TODO

lab.check("kalman", KalmanCV)

In [ ]:
def simulate(seed=0, n=400, dt=0.01, q_true=4.0, r_true=0.05 ** 2):
    rng = np.random.default_rng(seed)
    p = v = 0.0
    truth, meas = [], []
    for _ in range(n):
        v += rng.normal(0, np.sqrt(q_true * dt))
        p += v * dt
        truth.append(p); meas.append(p + rng.normal(0, np.sqrt(r_true)))
    return np.array(truth), np.array(meas)

try:
    truth, z = simulate()
    kf, est = KalmanCV(0.01, 4.0, 0.05 ** 2), []
    for zk in z:
        kf.predict(); kf.update(zk); est.append(kf.x[0])
    plt.plot(z, ".", ms=2, label="measured"); plt.plot(truth, label="truth"); plt.plot(est, label="Kalman")
    plt.legend(); plt.show()
except NotImplementedError:
    print("finish Task 1 first")

## Checkpoint: Math
**M1 (Apply).** Scalar filter: predicted variance $P = 4$, measurement variance $R = 1$. Kalman gain?

**M2 (Apply).** Variance after that update?

**M3 (Apply).** Predicted position 4.2 m, measurement 5.0 m. Innovation in meters?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** Over a long run the average NIS is 12 instead of about 1. What does it mean?
- a) the filter is too confident: Q (or R) is too small for the real errors
- b) the filter is too cautious
- c) nothing; NIS is not meaningful

**K2 (Evaluate).** Explain how you would tune Q and R for an encoder-plus-IMU joint velocity estimator on a real robot.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `e03.h`).
3. In `roboprep/checks/e03.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()